In [2]:
# Run this once to clear the cache
import shutil, os

cache = '/content/drive/MyDrive/dataset/oasis13d/src/__pycache__'
if os.path.exists(cache):
    shutil.rmtree(cache)
    print("✅ cache cleared")

# Then restart the Colab runtime:
# Runtime menu → Restart Runtime (Ctrl+M .)

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.insert(0, '/content/drive/MyDrive/dataset/oasis13d')

Mounted at /content/drive


In [4]:
# Double check Python actually sees the path
import sys
print([p for p in sys.path if 'oasis' in p])
# Must print: ['/content/drive/MyDrive/dataset/oasis13d']

['/content/drive/MyDrive/dataset/oasis13d']


In [5]:
from src.dataset import build_subject_list, OASISDataset
print("✅")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅


In [6]:
from src.preprocessing import (
    ClinicalPreprocessor, compute_class_weights,
    N_FEATURES, N_CLASSES, FEATURE_COLS, CDR_TO_CLASS
)
print("✅ preprocessing imported")

✅ preprocessing imported


In [8]:
from src.encoder_clinical import ClinicalMLPEncoder
import torch

encoder = ClinicalMLPEncoder()
dummy = torch.randn(4, 7)
out = encoder(dummy)
print(f"✅ encoder_clinical works — output: {tuple(out.shape)}")

✅ encoder_clinical works — output: (4, 1, 256)


In [9]:
from src.encoder_mri import BrainMRI3DEncoder
import torch

encoder = BrainMRI3DEncoder()
dummy = torch.randn(2, 1, 96, 96, 96)
patches, global_vec = encoder(dummy)
print(f"✅ encoder_mri works — patches: {tuple(patches.shape)}, global: {tuple(global_vec.shape)}")

✅ encoder_mri works — patches: (2, 216, 256), global: (2, 256)


In [10]:
import os

mri_root = '/content/drive/MyDrive/dataset/oasis13d/data'

for root, dirs, files in os.walk(mri_root):
    # Only show first 3 subjects to avoid flooding output
    nii_files = [f for f in files if f.endswith('.nii') or f.endswith('.nii.gz')]
    if nii_files:
        print(root)
        for f in nii_files[:2]:
            print(f'    {f}')
        break  # remove this break to see all

# Also print top-level folders
print('\nTop-level folders:')
for item in sorted(os.listdir(mri_root))[:10]:
    print(f'  {item}')

/content/drive/MyDrive/dataset/oasis13d/data/oasis/OASIS
    OAS1_0001_MR1_mpr_n4_anon_sbj_111_normalised.nii
    OAS1_0003_MR1_mpr_n4_anon_sbj_111_normalised.nii

Top-level folders:
  oasis
  oasis_cross-sectional.csv
  oasis_cross-sectional_facts.pdf


In [11]:
import os

mri_root = '/content/drive/MyDrive/dataset/oasis13d/data'

all_files = sorted(os.listdir(mri_root))

# Show first 10 files
print("First 10 files:")
for f in all_files[:10]:
    print(f'  {f}')

print(f"\nTotal files: {len(all_files)}")

First 10 files:
  oasis
  oasis_cross-sectional.csv
  oasis_cross-sectional_facts.pdf

Total files: 3


In [12]:
!pip install nibabel -q

from src.dataset import build_subject_list, OASISDataset
from src.preprocessing import ClinicalPreprocessor, CDR_TO_CLASS
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

BASE    = '/content/drive/MyDrive/dataset/oasis13d/data'
CSV     = f'{BASE}/oasis_cross-sectional.csv'
NII_DIR = f'{BASE}/oasis/OASIS'

df     = build_subject_list(CSV, NII_DIR)
labels = df['CDR'].map(CDR_TO_CLASS).values
df_tr, df_va = train_test_split(df, test_size=0.2, random_state=42, stratify=labels)

prep = ClinicalPreprocessor()
prep.fit_transform(df_tr)

train_ds = OASISDataset(df_tr, prep, augment=True)
loader   = DataLoader(train_ds, batch_size=2, shuffle=True)

mri, clinical, label = next(iter(loader))
print(f"MRI      : {tuple(mri.shape)}")
print(f"Clinical : {tuple(clinical.shape)}")
print(f"Label    : {label.tolist()}")

NIfTI scanner   : found 416 subjects in /content/drive/MyDrive/dataset/oasis13d/data/oasis/OASIS
Subject list    : 416 subjects with both CSV and NIfTI
Feature matrix : (332, 7)  (N_samples × 7 features)
Features       : ['Age', 'M/F', 'Educ', 'SES', 'MMSE', 'eTIV', 'nWBV']
Value range    : [-4.84, 3.22]  (after StandardScaler)
Class distribution:
  class 0 (Non-demented): ████████████████████████████████████████████████████████████ 252
  class 1 (Very mild): ████████████████████████████████████████████████████████ 56
  class 2 (Mild/Moderate): ████████████████████████ 24
OASISDataset    : 332 subjects  | augment=True  | shape=(96, 96, 96)
MRI      : (2, 1, 96, 96, 96)
Clinical : (2, 7)
Label    : [0, 0]


In [5]:
from src.fusion import CrossModalFusion, attention_to_volume
import torch

D, B = 256, 4
patches    = torch.randn(B, 216, D)
global_vec = torch.randn(B, D)
clin_token = torch.randn(B, 1, D)

fusion = CrossModalFusion()
fused, attn_w = fusion(patches, global_vec, clin_token)

print(f"fused    : {tuple(fused.shape)}")     # (4, 256)
print(f"attn_w   : {tuple(attn_w.shape)}")    # (4, 8, 1, 216)
print("✅ fusion works")

fused    : (4, 256)
attn_w   : (4, 8, 1, 216)
✅ fusion works
